# CC training sperimentale

In [1]:
# Decommentare Whitelib import quando si gira sul chip.

#from qutip import *
import numpy as np
import matplotlib.pyplot as plt
#import perceval as pcvl
from collections import Counter
import time
import scipy.linalg
from tqdm import tqdm
# from WhiteLib import change_voltages, data_collection

In [2]:
# Varie funzioni generali.

def MyHamiltonian(paramsTrain, paramsNoTrain, sizeHam):
    #return MyHamiltonianDiagonal(paramsTrain, paramsNoTrain, sizeHam)
    return MyHamiltonianHoneycomb(paramsTrain, paramsNoTrain, sizeHam)

def MyUnitary(paramsTrain, paramsNoTrain, timeEval, sizeHam, paramsUnitary = []):
    #return MyUnitarySingle(paramsTrain, paramsNoTrain, timeEval, sizeHam)
    return MyUnitarySegmented(paramsTrain, paramsNoTrain, timeEval, sizeHam, paramsUnitary)


    


def MyMae(y_predicted, y_true):
    total_error_arr = 0
    for yp, yt in zip(y_predicted, y_true):
        total_error_arr += abs(yp - yt)
    total_error = np.sum(total_error_arr)
    #print("Total error is:",total_error)
    mae = total_error/len(y_predicted)
    #print("Mean absolute error is:",mae)
    return mae


def createMeshes32(modeCoordinates, multiplier):
    alterationMeshes = np.zeros([2,32])
    for j in range(2):
        for i in range(32):
            distance = 1 + np.linalg.norm(modeCoordinates[i]-modeCoordinates[j*7])
            alterationMeshes[j][i] = 1 - (np.log(distance) * multiplier)
            if (alterationMeshes[j][i] < 0):
                alterationMeshes[j][i] = 0
    return (alterationMeshes)


def MyHamiltonianHoneycomb(paramsTrain, paramsNoTrain, sizeHam):
    tempArr = np.zeros([sizeHam,sizeHam])
    tempCounter = 0
    for i in range(sizeHam):
        j = i
        tempArr[i,j] = paramsTrain[i]
        if (i % 8 != 0):
            j = i-1
            tempArr[i,j] = paramsNoTrain[(tempCounter)]
            tempArr[j,i] = paramsNoTrain[(tempCounter)]
            #print(tempCounter)
            tempCounter += 1
        if ((i//8)%2 == 0):
            if (i > 7):
                j = i-8
                tempArr[i,j] = paramsNoTrain[(tempCounter)]
                tempArr[j,i] = paramsNoTrain[(tempCounter)]
                tempCounter += 1
                if (i % 8 != 0):
                    j = i-9
                    tempArr[i,j] = paramsNoTrain[(tempCounter)]
                    tempArr[j,i] = paramsNoTrain[(tempCounter)]
                    tempCounter += 1
            if ((i+8) < sizeHam):
                j = i+8
                tempArr[i,j] = paramsNoTrain[(tempCounter)]
                tempArr[j,i] = paramsNoTrain[(tempCounter)]   
                tempCounter += 1
                if (i % 8 != 0):
                    j = i+7
                    tempArr[i,j] = paramsNoTrain[(tempCounter)]
                    tempArr[j,i] = paramsNoTrain[(tempCounter)]
                    tempCounter += 1
    tempHam = Qobj(tempArr)
    #return(tempArr)
    return(tempHam)


def MyHamiltonianDiagonal(paramsTrain, paramsNoTrain, sizeHam):
    tempArr = np.zeros([sizeHam,sizeHam])
    for i in range(sizeHam):
        for j in range(sizeHam):
            if (j+1 == i):
                tempArr[i,j] = paramsNoTrain[(i-1)]
            if (j == i):
                tempArr[i,j] = paramsTrain[i]
            if (j-1 == i):
                tempArr[i,j] = paramsNoTrain[(i)]
    tempHam = Qobj(tempArr)
    return(tempHam)


def MyUnitarySingle(paramsTrain, paramsNoTrain, timeEval, sizeHam):
    hamOptions = {
      #"atol": 1e-11,
      #"rtol": 1e-10,
      "atol": 1e-14,
      "rtol": 1e-13,
      "nsteps": 10000
    }
    tempHamiltonian = MyHamiltonian(paramsTrain, paramsNoTrain, sizeHam)
    tempUnitary = qutip.propagator(tempHamiltonian, [0, timeEval], options = hamOptions)
    #print(tempUnitary)
    unitaryToUse = tempUnitary[1].full()
    #print(unitaryToUse)
    #print(tempUnitary[1] * tempUnitary[1].dag())
    matrixPerc = pcvl.Matrix(unitaryToUse, use_symbolic = False)
    return matrixPerc


def ParameterConversion(paramsTrainModifier, sizeHam, baseTrain, alterationMeshes, numSegments, tweakableSegments):
    numParametersTemp = len(baseTrain)
    paramsTrainArray = np.zeros((numSegments, numParametersTemp))
    j = 0
    for i in range(numSegments):
        if (tweakableSegments[i] == 0):
            paramsTrainArray[i] = baseTrain
        if (tweakableSegments[i] == 1):
            paramsTrainArray[i] = baseTrain + (paramsTrainModifier[j] * alterationMeshes[0]) + (paramsTrainModifier[j+1] * alterationMeshes[1])
            j = j + 2
    return paramsTrainArray


def MyUnitarySegmented(paramsTrainModifier, paramsNoTrain, timeEval, sizeHam, paramsUnitary):
    # make for loop with multiple calls at MyUnitarySingle, and then compile them together (using np.dot)
    baseTrain = paramsUnitary["baseTrain"]
    alterationMeshes = paramsUnitary["alterationMeshes"]
    numSegments = paramsUnitary["numSegments"]
    timeSegments = paramsUnitary["timeSegments"]
    tweakableSegments = paramsUnitary["tweakableSegments"]
    paramsTrainArray = ParameterConversion(paramsTrainModifier, sizeHam, baseTrain, alterationMeshes, numSegments, tweakableSegments)
    unitaryAssembled = MyUnitarySingle(paramsTrainArray[0], paramsNoTrain, timeSegments, sizeHam)
    for i in range((numSegments-1)):
        unitaryTemp = MyUnitarySingle(paramsTrainArray[i+1], paramsNoTrain, timeSegments, sizeHam)
        unitaryAssembledTemp = np.dot(unitaryAssembled, unitaryTemp)
        unitaryAssembled = unitaryAssembledTemp.copy()
    return unitaryAssembled


def MyQuantumCircSim(paramsTrain, paramsNoTrain, timeEval, sizeHam, inputStates, outputStates, shotSize, paramsUnitary = []):
    tempMatrix = MyUnitary(paramsTrain, paramsNoTrain, timeEval, sizeHam, paramsUnitary)
    tempUnitaryPerc = pcvl.components.Unitary(tempMatrix)
    QPU = pcvl.Processor("SLOS", tempUnitaryPerc)
    tempResults = np.zeros([len(inputStates), len(outputStates)])
    for i in range(len(inputStates)): 
        QPU.with_input(inputStates[i])
        tempSampler = pcvl.algorithm.Sampler(QPU)
        tempSampleCount = tempSampler.sample_count(shotSize)
        tempCalculatedProbs = tempSampleCount["results"]
        for j in range(len(outputStates)):
            tempResults[i, j] = tempCalculatedProbs[outputStates[j]]
        #print(potatoSamples)
    tempProbs = tempResults/shotSize
    return tempProbs


def MyQuantumCircSimPerfect(paramsTrain, paramsNoTrain, timeEval, sizeHam, inputStates, outputStates, paramsUnitary = []):
    tempMatrix = MyUnitary(paramsTrain, paramsNoTrain, timeEval, sizeHam, paramsUnitary)
    tempUnitaryPerc = pcvl.components.Unitary(tempMatrix)
    QPU = pcvl.Processor("SLOS", tempUnitaryPerc)
    #print("The input state: ", input_state)
    #analyzer= pcvl.algorithm.Analyzer(QPU, [input_state], '*')
    analyzer= pcvl.algorithm.Analyzer(QPU, inputStates, outputStates)
    tempResults = analyzer.distribution
    #return tempResults
    return np.array(tempResults.real)


def MyQuantumCircSimUnitaryPerfect(chipUnitary, inputStates, outputStates, paramsUnitary = []):
    matrixPerc = pcvl.Matrix(chipUnitary, use_symbolic = False)
    tempUnitaryPerc = pcvl.components.Unitary(matrixPerc)
    QPU = pcvl.Processor("SLOS", tempUnitaryPerc)
    #print("The input state: ", input_state)
    #analyzer= pcvl.algorithm.Analyzer(QPU, [input_state], '*')
    analyzer= pcvl.algorithm.Analyzer(QPU, inputStates, outputStates)
    tempResults = analyzer.distribution
    #return tempResults
    return np.array(tempResults.real)


def lossEval(paramsTrain, paramsNoTrain, timeEval, sizeHam, inputStates, outputStates, target, shotSize, perfectComp = False, paramsUnitary = []):
    if (perfectComp == False):
        tempPrediction = MyQuantumCircSim(paramsTrain, paramsNoTrain, timeEval, sizeHam, inputStates, outputStates, shotSize, paramsUnitary)
    else:
        tempPrediction = MyQuantumCircSimPerfect(paramsTrain, paramsNoTrain, timeEval, sizeHam, inputStates, outputStates, paramsUnitary)
    tempLoss = MyMae(tempPrediction, target)
    return(tempLoss)


def MyFidelity(paramsTrainCurrent, paramsNoTrainCurrent, paramsTrainTarget, paramsNoTrainTarget, timeEval, sizeHam, paramsUnitary1 = [], paramsUnitary2 = []):
    hamOptions = {
      #"atol": 1e-11,
      #"rtol": 1e-10,
      "atol": 1e-14,
      "rtol": 1e-13,
      "nsteps": 10000
    }
    unitaryToUseCurrent = MyUnitary(paramsTrainCurrent, paramsNoTrainCurrent, timeEval, sizeHam, paramsUnitary1)
    unitaryToUseTarget = MyUnitary(paramsTrainTarget, paramsNoTrainTarget, timeEval, sizeHam, paramsUnitary2)
    tempFidelity = (1./sizeHam)*np.abs(np.trace(np.dot(unitaryToUseCurrent, np.transpose(np.conjugate(unitaryToUseTarget)))))
    #return tempFidelity, unitaryToUseCurrent, unitaryToUseTarget
    return tempFidelity


def MyFidelityUnitaryTarget(paramsTrainCurrent, paramsNoTrainCurrent, targetUnitary, timeEval, sizeHam, paramsUnitary1 = [], paramsUnitary2 = []):
    hamOptions = {
      #"atol": 1e-11,
      #"rtol": 1e-10,
      "atol": 1e-14,
      "rtol": 1e-13,
      "nsteps": 10000
    }
    unitaryToUseCurrent = MyUnitary(paramsTrainCurrent, paramsNoTrainCurrent, timeEval, sizeHam, paramsUnitary1)
    unitaryToUseTarget = targetUnitary
    tempFidelity = (1./sizeHam)*np.abs(np.trace(np.dot(unitaryToUseCurrent, np.transpose(np.conjugate(unitaryToUseTarget)))))
    #return tempFidelity, unitaryToUseCurrent, unitaryToUseTarget
    return tempFidelity


def MyFidelityUnitaries(unitaryToUseCurrent, unitaryToUseTarget):
    tempFidelity = (1./sizeHam)*np.abs(np.trace(np.dot(unitaryToUseCurrent, np.transpose(np.conjugate(unitaryToUseTarget)))))
    #return tempFidelity, unitaryToUseCurrent, unitaryToUseTarget
    return tempFidelity


def UpdateParameter(currentValue, shiftValue, avoidBoundary = False):
    currentValue = currentValue + shiftValue
    if (avoidBoundary == False):
        if (currentValue > parameterValueMax):
            currentValue = parameterValueMax
        if (currentValue < parameterValueMin):
            currentValue = parameterValueMin
    if (avoidBoundary == True):
        if (currentValue > parameterValueMax):
            currentValue = parameterValueMaxReset
        if (currentValue < parameterValueMin):
            currentValue = parameterValueMinReset
    return currentValue

In [3]:
# Varie funzioni training.

def myTrainingLoopPartial(currentParamsTrainable, paramsNotTrainable, baseParamsTrainable, timeCoupling, sizeHam, numParams, input_states_one, output_states_one, targetState1, shotSize1, input_states_two_full, output_states_two, targetState2_full, shotSize2, trainingParams, paramsUnitary = []):
    epochsNum = trainingParams["epochsNum"]
    LR_check = trainingParams["LR_check"]
    LR_move = trainingParams["LR_move"]
    useTwoPhotons = trainingParams["useTwoPhotons"]
    typeTraining = trainingParams["typeTraining"]
    typeOrder = trainingParams["typeOrder"]
    printProgress = trainingParams["printProgress"]
    checkPairsNum = trainingParams["checkPairsNum"]
    firstNeighbourList = trainingParams["firstNeighbourList"]
    avoidBoundary = trainingParams["avoidBoundary"]
    paramsOrderList = np.arange(numParams)
    bestLoss = 1000
    maxPairs = len(input_states_two_full)
    bestParams = np.zeros_like(baseParamsTrainable)
    lossHistory = np.empty(epochsNum + 1)
    fidelityHistory = np.empty(epochsNum + 1)
    
    for i in range(epochsNum):
        if (typeOrder == "allRandom"):
            chosenParam = np.random.choice(numParams)
        elif (typeOrder == "listRandom"):
            if (i%numParams == 0):
                np.random.shuffle(paramsOrderList)
            chosenParam = paramsOrderList[i%numParams]
        else:
            print("ERROR, NO VALID ORDER TYPE SELECTED")

        chosenPairs = np.random.choice(maxPairs, checkPairsNum, replace=False)
        input_states_two = list( input_states_two_full[i] for i in chosenPairs )
        targetState2 = targetState2_full[chosenPairs]
        #print(input_states_two)
        
        currentFidelity = MyFidelity(currentParamsTrainable, paramsNotTrainable, baseParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, paramsUnitary, paramsUnitary)
        prevLoss = lossEval(currentParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, input_states_one, output_states_one, targetState1, shotSize1, paramsUnitary = paramsUnitary)     #can be skipped if training step is equal to check step.
        if (useTwoPhotons == True):
            prevLoss = prevLoss + lossEval(currentParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, input_states_two, output_states_two, targetState2, shotSize2, paramsUnitary = paramsUnitary)
        if (printProgress == "all"):
            print("Epoch:", i)
            print("Current loss is:", prevLoss, "    Current fidelity is:", currentFidelity, "    Changed param:", chosenParam)
            print(chosenPairs)
        #print("Current fidelity is:", currentFidelity) 
        #print("Changed param:", chosenParam)
        if (prevLoss < bestLoss):
            bestLoss = prevLoss
            bestParams = currentParamsTrainable.copy()
        lossHistory[i] = prevLoss
        fidelityHistory[i] = currentFidelity
        
        
        checkShift = prevLoss * LR_check
        tempStore = currentParamsTrainable[chosenParam].copy()
        currentParamsTrainable[chosenParam] = UpdateParameter(tempStore, checkShift, avoidBoundary)
        upLoss = lossEval(currentParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, input_states_one, output_states_one, targetState1, shotSize1, paramsUnitary = paramsUnitary)
        if (useTwoPhotons == True):
            upLoss = upLoss + lossEval(currentParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, input_states_two, output_states_two, targetState2, shotSize2, paramsUnitary = paramsUnitary)
        currentParamsTrainable[chosenParam] = UpdateParameter(tempStore, (-checkShift), avoidBoundary)
        downLoss = lossEval(currentParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, input_states_one, output_states_one, targetState1, shotSize1, paramsUnitary = paramsUnitary)
        if (useTwoPhotons == True):
            downLoss = downLoss + lossEval(currentParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, input_states_two, output_states_two, targetState2, shotSize2, paramsUnitary = paramsUnitary)
        # calculating which direction to move
        if ((upLoss < prevLoss) & (downLoss < prevLoss)):
            if (downLoss < upLoss):
                proportion = -1 * ((prevLoss - downLoss)/prevLoss)
            else:
                proportion = ((prevLoss - upLoss)/prevLoss)
        elif(upLoss < prevLoss):
            proportion = ((prevLoss - upLoss)/prevLoss)
        elif(downLoss < prevLoss):
            proportion = -1 * ((prevLoss - downLoss)/prevLoss)
        else:
            if (printProgress == "all"):
                print("Changing parameter value does not improve loss")
            proportion = 0
        # moving in the decided direction based on training type
        if (typeTraining == "proportional"):
            currentParamsTrainable[chosenParam] = UpdateParameter(tempStore, (proportion*LR_move), avoidBoundary)
        elif (typeTraining == "absolute"):
            currentParamsTrainable[chosenParam] = UpdateParameter(tempStore, (np.sign(proportion)*checkShift*LR_move/LR_check), avoidBoundary)
        else:
            print("ERROR, NO VALID TRAINING TYPE SELECTED")
    
    prevLoss = lossEval(currentParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, input_states_one, output_states_one, targetState1, shotSize1, paramsUnitary = paramsUnitary)
    if (useTwoPhotons == True):
        prevLoss = prevLoss + lossEval(currentParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, input_states_two_full, output_states_two, targetState2_full, shotSize2, paramsUnitary = paramsUnitary)    
    currentFidelity = MyFidelity(currentParamsTrainable, paramsNotTrainable, baseParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, paramsUnitary, paramsUnitary)
    lossHistory[-1] = prevLoss
    fidelityHistory[-1] = currentFidelity
    if (printProgress == "last" or printProgress == "all"):
        print("Last loss is:", prevLoss, "    Last fidelity is:", currentFidelity)
    return currentParamsTrainable, lossHistory, fidelityHistory, bestParams, bestLoss


def myTrainingLoop(currentParamsTrainable, paramsNotTrainable, targetParamsTrainable, timeCoupling, sizeHam, numParams, input_states_one, output_states_one, targetState1, shotSize1, input_states_two, output_states_two, targetState2, shotSize2, trainingParams, paramsUnitary = []):
    epochsNum = trainingParams["epochsNum"]
    LR_check = trainingParams["LR_check"]
    LR_move = trainingParams["LR_move"]
    useTwoPhotons = trainingParams["useTwoPhotons"]
    typeTraining = trainingParams["typeTraining"]
    typeOrder = trainingParams["typeOrder"]
    printProgress = trainingParams["printProgress"]
    paramsOrderList = np.arange(numParams)
    prevLoss = lossEval(currentParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, input_states_one, output_states_one, targetState1, shotSize1, paramsUnitary = paramsUnitary)
    if (useTwoPhotons == True):
        prevLoss = prevLoss + lossEval(currentParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, input_states_two, output_states_two, targetState2, shotSize2, paramsUnitary = paramsUnitary)
    bestLoss = 1000
    bestParams = np.zeros_like(targetParamsTrainable)
    lossHistory = np.empty(epochsNum)
    fidelityHistory = np.empty(epochsNum)
    for i in range(epochsNum):
        if (typeOrder == "allRandom"):
            chosenParam = np.random.choice(numParams)
        elif (typeOrder == "listRandom"):
            if (i%numParams == 0):
                np.random.shuffle(paramsOrderList)
            chosenParam = paramsOrderList[i%numParams]
        else:
            print("ERROR, NO VALID ORDER TYPE SELECTED")
        checkShift = prevLoss * LR_check
        tempStore = currentParamsTrainable[chosenParam].copy()
        currentParamsTrainable[chosenParam] = UpdateParameter(tempStore, checkShift)
        upLoss = lossEval(currentParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, input_states_one, output_states_one, targetState1, shotSize1, paramsUnitary = paramsUnitary)
        if (useTwoPhotons == True):
            upLoss = upLoss + lossEval(currentParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, input_states_two, output_states_two, targetState2, shotSize2, paramsUnitary = paramsUnitary)
        currentParamsTrainable[chosenParam] = UpdateParameter(tempStore, (-checkShift))
        downLoss = lossEval(currentParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, input_states_one, output_states_one, targetState1, shotSize1, paramsUnitary = paramsUnitary)
        if (useTwoPhotons == True):
            downLoss = downLoss + lossEval(currentParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, input_states_two, output_states_two, targetState2, shotSize2, paramsUnitary = paramsUnitary)
        # calculating which direction to move
        if ((upLoss < prevLoss) & (downLoss < prevLoss)):
            if (downLoss < upLoss):
                proportion = -1 * ((prevLoss - downLoss)/prevLoss)
            else:
                proportion = ((prevLoss - upLoss)/prevLoss)
        elif(upLoss < prevLoss):
            proportion = ((prevLoss - upLoss)/prevLoss)
        elif(downLoss < prevLoss):
            proportion = -1 * ((prevLoss - downLoss)/prevLoss)
        else:
            if (printProgress == "all"):
                print("Changing parameter value does not improve loss")
            proportion = 0
        # moving in the decided direction based on training type
        if (typeTraining == "proportional"):
            currentParamsTrainable[chosenParam] = UpdateParameter(tempStore, (proportion*LR_move))
        elif (typeTraining == "absolute"):
            currentParamsTrainable[chosenParam] = UpdateParameter(tempStore, (np.sign(proportion)*checkShift*LR_move/LR_check))
        else:
            print("ERROR, NO VALID TRAINING TYPE SELECTED")
        prevLoss = lossEval(currentParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, input_states_one, output_states_one, targetState1, shotSize1, paramsUnitary = paramsUnitary)     #can be skipped if training step is equal to check step.
        if (useTwoPhotons == True):
            prevLoss = prevLoss + lossEval(currentParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, input_states_two, output_states_two, targetState2, shotSize2, paramsUnitary = paramsUnitary)
    
        currentFidelity = MyFidelity(currentParamsTrainable, paramsNotTrainable, targetParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, paramsUnitary, paramsUnitary)
        lossHistory[i] = prevLoss
        fidelityHistory[i] = currentFidelity
        if (printProgress == "all"):
            print("Epoch:", i)
            print("Current loss is:", prevLoss, "    Current fidelity is:", currentFidelity, "    Changed param:", chosenParam)
        #print("Current fidelity is:", currentFidelity) 
        #print("Changed param:", chosenParam)
        if (prevLoss < bestLoss):
            bestLoss = prevLoss
            bestParams = currentParamsTrainable.copy()
    if (printProgress == "last"):
        print("Last loss is:", prevLoss, "    Last fidelity is:", currentFidelity)
    return currentParamsTrainable, lossHistory, fidelityHistory, bestParams, bestLoss


def myTrainingLoopPartialUnitaryTarget(currentParamsTrainable, paramsNotTrainable, targetUnitary, timeCoupling, sizeHam, numParams, input_states_one, output_states_one, targetState1, shotSize1, input_states_two_full, output_states_two, targetState2_full, shotSize2, trainingParams, paramsUnitary = []):
    epochsNum = trainingParams["epochsNum"]
    LR_check = trainingParams["LR_check"]
    LR_move = trainingParams["LR_move"]
    useTwoPhotons = trainingParams["useTwoPhotons"]
    typeTraining = trainingParams["typeTraining"]
    typeOrder = trainingParams["typeOrder"]
    printProgress = trainingParams["printProgress"]
    checkPairsNum = trainingParams["checkPairsNum"]
    firstNeighbourList = trainingParams["firstNeighbourList"]
    avoidBoundary = trainingParams["avoidBoundary"]
    paramsOrderList = np.arange(numParams)
    bestLoss = 1000
    maxPairs = len(input_states_two_full)
    bestParams = np.zeros_like(currentParamsTrainable)
    lossHistory = np.empty(epochsNum + 1)
    fidelityHistory = np.empty(epochsNum + 1)
    
    for i in range(epochsNum):
        if (typeOrder == "allRandom"):
            chosenParam = np.random.choice(numParams)
        elif (typeOrder == "listRandom"):
            if (i%numParams == 0):
                np.random.shuffle(paramsOrderList)
            chosenParam = paramsOrderList[i%numParams]
        else:
            print("ERROR, NO VALID ORDER TYPE SELECTED")

        chosenPairs = np.random.choice(maxPairs, checkPairsNum, replace=False)
        input_states_two = list( input_states_two_full[i] for i in chosenPairs )
        targetState2 = targetState2_full[chosenPairs]
        #print(input_states_two)
        
        currentFidelity = MyFidelityUnitaryTarget(currentParamsTrainable, paramsNotTrainable, targetUnitary, timeCoupling, sizeHam, paramsUnitary, paramsUnitary)
        prevLoss = lossEval(currentParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, input_states_one, output_states_one, targetState1, shotSize1, paramsUnitary = paramsUnitary)     #can be skipped if training step is equal to check step.
        if (useTwoPhotons == True):
            prevLoss = prevLoss + lossEval(currentParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, input_states_two, output_states_two, targetState2, shotSize2, paramsUnitary = paramsUnitary)
        if (printProgress == "all"):
            print("Epoch:", i)
            print("Current loss is:", prevLoss, "    Current fidelity is:", currentFidelity, "    Changed param:", chosenParam)
            print(chosenPairs)
        #print("Current fidelity is:", currentFidelity) 
        #print("Changed param:", chosenParam)
        if (prevLoss < bestLoss):
            bestLoss = prevLoss
            bestParams = currentParamsTrainable.copy()
        lossHistory[i] = prevLoss
        fidelityHistory[i] = currentFidelity
        
        
        checkShift = prevLoss * LR_check
        tempStore = currentParamsTrainable[chosenParam].copy()
        currentParamsTrainable[chosenParam] = UpdateParameter(tempStore, checkShift, avoidBoundary)
        upLoss = lossEval(currentParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, input_states_one, output_states_one, targetState1, shotSize1, paramsUnitary = paramsUnitary)
        if (useTwoPhotons == True):
            upLoss = upLoss + lossEval(currentParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, input_states_two, output_states_two, targetState2, shotSize2, paramsUnitary = paramsUnitary)
        currentParamsTrainable[chosenParam] = UpdateParameter(tempStore, (-checkShift), avoidBoundary)
        downLoss = lossEval(currentParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, input_states_one, output_states_one, targetState1, shotSize1, paramsUnitary = paramsUnitary)
        if (useTwoPhotons == True):
            downLoss = downLoss + lossEval(currentParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, input_states_two, output_states_two, targetState2, shotSize2, paramsUnitary = paramsUnitary)
        # calculating which direction to move
        if ((upLoss < prevLoss) & (downLoss < prevLoss)):
            if (downLoss < upLoss):
                proportion = -1 * ((prevLoss - downLoss)/prevLoss)
            else:
                proportion = ((prevLoss - upLoss)/prevLoss)
        elif(upLoss < prevLoss):
            proportion = ((prevLoss - upLoss)/prevLoss)
        elif(downLoss < prevLoss):
            proportion = -1 * ((prevLoss - downLoss)/prevLoss)
        else:
            if (printProgress == "all"):
                print("Changing parameter value does not improve loss")
            proportion = 0
        # moving in the decided direction based on training type
        if (typeTraining == "proportional"):
            currentParamsTrainable[chosenParam] = UpdateParameter(tempStore, (proportion*LR_move), avoidBoundary)
        elif (typeTraining == "absolute"):
            currentParamsTrainable[chosenParam] = UpdateParameter(tempStore, (np.sign(proportion)*checkShift*LR_move/LR_check), avoidBoundary)
        else:
            print("ERROR, NO VALID TRAINING TYPE SELECTED")
    
    prevLoss = lossEval(currentParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, input_states_one, output_states_one, targetState1, shotSize1, paramsUnitary = paramsUnitary)
    if (useTwoPhotons == True):
        prevLoss = prevLoss + lossEval(currentParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, input_states_two_full, output_states_two, targetState2_full, shotSize2, paramsUnitary = paramsUnitary)    
    currentFidelity = MyFidelityUnitaryTarget(currentParamsTrainable, paramsNotTrainable, targetUnitary, timeCoupling, sizeHam, paramsUnitary, paramsUnitary)
    lossHistory[-1] = prevLoss
    fidelityHistory[-1] = currentFidelity
    if (printProgress == "last" or printProgress == "all"):
        print("Last loss is:", prevLoss, "    Last fidelity is:", currentFidelity)
    return currentParamsTrainable, lossHistory, fidelityHistory, bestParams, bestLoss

In [4]:
# Funzioni modificate per il caso sperimentale.

def lossEvalExp(parameters, inputs, target, duration, repetitions_singles, repetitions_doubles, supply, Nsupp, boxes, exposition):
    tempPrediction = data_collection(inputs, np.sqrt(parameters), supply, Nsupp, boxes, exposition, duration, repetitions_singles, repetitions_doubles)
    tempLoss = MyMaeExp(tempPrediction, target)
    #print(tempLoss)
    return(tempLoss)

def MyMaeExp(y_predicted, y_true):
    total_error_arr = 0
    for yp, yt in zip(y_predicted, y_true):
        yp = (yp/np.sum(yp))
        yt = (yt/np.sum(yt))
        #print("yp: ", yp, "yt: ", yt)
        total_error_arr += abs(yp - yt)
    total_error = np.sum(total_error_arr)
    #print("Total error is:",total_error)
    mae = total_error/len(y_predicted)
    #print("Mean absolute error is:",mae)
    return mae


In [5]:
def UpdateParameter(currentValue, shiftValue, avoidBoundary, parameterValueMin, parameterValueMax, parameterValueMaxReset, parameterValueMinReset):
    currentValue = currentValue + shiftValue
    if (avoidBoundary == False):
        if (currentValue > parameterValueMax):
            currentValue = parameterValueMax
        if (currentValue < parameterValueMin):
            currentValue = parameterValueMin
    if (avoidBoundary == True):
        if (currentValue > parameterValueMax):
            currentValue = parameterValueMaxReset
        if (currentValue < parameterValueMin):
            currentValue = parameterValueMinReset
    return currentValue

In [6]:
def myTrainingLoopExp(currentParamsTrainable, duration, repetitions_singles, repetitions_doubles, numParams, input_states_one, targetState1, input_states_two_full, targetState2_full, trainingParams, paramsUnitary = []):
    epochsNum = trainingParams["epochsNum"]
    LR_check = trainingParams["LR_check"]
    LR_move = trainingParams["LR_move"]
    useTwoPhotons = trainingParams["useTwoPhotons"]
    typeTraining = trainingParams["typeTraining"]
    typeOrder = trainingParams["typeOrder"]
    printProgress = trainingParams["printProgress"]
    checkPairsNum = trainingParams["checkPairsNum"]
    firstNeighbourList = trainingParams["firstNeighbourList"]
    avoidBoundary = trainingParams["avoidBoundary"]
    supply = trainingParams["supply"]
    Nsupp = trainingParams["Nsupp"]
    boxes = trainingParams["boxes"]
    exposition= trainingParams["exposition"]
    parameterValueMin = trainingParams["parameterValueMin"]
    parameterValueMax = trainingParams["parameterValueMax"]
    parameterValueMaxReset = trainingParams["parameterValueMaxReset"]
    parameterValueMinReset = trainingParams["parameterValueMinReset"]
    paramsOrderList = np.arange(numParams)
    bestLoss = 1000
    maxPairs = len(input_states_two_full)
    bestParams = np.zeros_like(currentParamsTrainable)
    lossHistory = np.empty(epochsNum + 1)
    #fidelityHistory = np.empty(epochsNum + 1)
    
    for epoch in range(epochsNum):
        if (typeOrder == "allRandom"):
            chosenParam = np.random.choice(numParams)
        elif (typeOrder == "listRandom"):
            if (epoch%numParams == 0):
                np.random.shuffle(paramsOrderList)
            chosenParam = paramsOrderList[epoch%numParams]
        else:
            print("ERROR, NO VALID ORDER TYPE SELECTED")

        chosenPairs = np.random.choice(maxPairs, checkPairsNum, replace=False)
        input_states_two = list( input_states_two_full[j] for j in chosenPairs )
        targetState2 = np.zeros((checkPairsNum, len(targetState2_full[0])))
        for j in range(checkPairsNum):
            targetState2[j] = targetState2_full[chosenPairs[j]]

        #print(input_states_two)
        
        #currentFidelity = MyFidelity(currentParamsTrainable, paramsNotTrainable, baseParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, paramsUnitary, paramsUnitary)
        
        prevLoss = lossEvalExp(currentParamsTrainable, input_states_one, targetState1, duration, repetitions_singles, repetitions_doubles, supply, Nsupp, boxes, exposition)     #can be skipped if training step is equal to check step.
        if (useTwoPhotons == True):
            prevLoss = prevLoss + lossEvalExp(currentParamsTrainable, input_states_two, targetState2, duration, repetitions_singles, repetitions_doubles, supply, Nsupp, boxes, exposition)
        if (printProgress == "all"):
            print("Epoch:", epoch)
            print("Current loss is:", prevLoss,  "    Changed param:", chosenParam)
            #print("Current loss is:", prevLoss, "    Current fidelity is:", currentFidelity, "    Changed param:", chosenParam)
            print(chosenPairs)
        #print("Current fidelity is:", currentFidelity) 
        #print("Changed param:", chosenParam)
        if (prevLoss < bestLoss):
            bestLoss = prevLoss
            bestParams = currentParamsTrainable.copy()
        lossHistory[epoch] = prevLoss
        #fidelityHistory[epoch] = currentFidelity
        
        
        checkShift = prevLoss * LR_check
        tempStore = currentParamsTrainable[chosenParam]
        currentParamsTrainable[chosenParam] = UpdateParameter(tempStore, checkShift, avoidBoundary, parameterValueMin, parameterValueMax, parameterValueMaxReset, parameterValueMinReset)
        #print(currentParamsTrainable)
        upLoss = lossEvalExp(currentParamsTrainable, input_states_one, targetState1, duration, repetitions_singles, repetitions_doubles, supply, Nsupp, boxes, exposition)
        if (useTwoPhotons == True):
            upLoss = upLoss + lossEvalExp(currentParamsTrainable, input_states_two, targetState2, duration, repetitions_singles, repetitions_doubles, supply, Nsupp, boxes, exposition)
        currentParamsTrainable[chosenParam] = UpdateParameter(tempStore, (-checkShift), avoidBoundary, parameterValueMin, parameterValueMax, parameterValueMaxReset, parameterValueMinReset)
        downLoss = lossEvalExp(currentParamsTrainable, input_states_one, targetState1, duration, repetitions_singles, repetitions_doubles, supply, Nsupp, boxes, exposition)
        if (useTwoPhotons == True):
            downLoss = downLoss + lossEvalExp(currentParamsTrainable, input_states_two, targetState2, duration, repetitions_singles, repetitions_doubles, supply, Nsupp, boxes, exposition)
        # calculating which direction to move
        if ((upLoss < prevLoss) & (downLoss < prevLoss)):
            if (downLoss < upLoss):
                proportion = -1 * ((prevLoss - downLoss)/prevLoss)
            else:
                proportion = ((prevLoss - upLoss)/prevLoss)
        elif(upLoss < prevLoss):
            proportion = ((prevLoss - upLoss)/prevLoss)
        elif(downLoss < prevLoss):
            proportion = -1 * ((prevLoss - downLoss)/prevLoss)
        else:
            if (printProgress == "all"):
                print("Changing parameter value does not improve loss")
            proportion = 0
        # moving in the decided direction based on training type
        if (typeTraining == "proportional"):
            currentParamsTrainable[chosenParam] = UpdateParameter(tempStore, (proportion*LR_move), avoidBoundary, parameterValueMin, parameterValueMax, parameterValueMaxReset, parameterValueMinReset)
        elif (typeTraining == "absolute"):
            currentParamsTrainable[chosenParam] = UpdateParameter(tempStore, (np.sign(proportion)*checkShift*LR_move/LR_check), avoidBoundary, parameterValueMin, parameterValueMax, parameterValueMaxReset, parameterValueMinReset)
        else:
            print("ERROR, NO VALID TRAINING TYPE SELECTED")
    
    prevLoss = lossEvalExp(currentParamsTrainable, input_states_one, targetState1, duration, repetitions_singles, repetitions_doubles, supply, Nsupp, boxes, exposition)
    if (useTwoPhotons == True):
            prevLoss = prevLoss + lossEvalExp(currentParamsTrainable, input_states_two, targetState2, duration, repetitions_singles, repetitions_doubles, supply, Nsupp, boxes, exposition)
    #currentFidelity = MyFidelity(currentParamsTrainable, paramsNotTrainable, baseParamsTrainable, paramsNotTrainable, timeCoupling, sizeHam, paramsUnitary, paramsUnitary)
    lossHistory[-1] = prevLoss
    #fidelityHistory[-1] = currentFidelity
    if (printProgress == "last" or printProgress == "all"):
        #print("Last loss is:", prevLoss, "    Last fidelity is:", currentFidelity)
        print("Last loss is:", prevLoss)
    #return currentParamsTrainable, lossHistory, fidelityHistory, bestParams, bestLoss
    return currentParamsTrainable, lossHistory, bestParams, bestLoss

In [7]:
#myTrainingLoopExp(currentParamsTrainable, duration, repetitions_singles, repetitions_doubles, numParams, input_states_one, targetSingles, input_states_two_full, targetDoubles, trainingParams)


In [8]:
# Definizione parametri fissi di device  training.

addresses = ['ASRL24::INSTR', 'ASRL25::INSTR', 'ASRL26::INSTR', 'ASRL27::INSTR', 'ASRL28::INSTR', 'ASRL29::INSTR','USB0::0x05E6::0x2230::9100018::INSTR', 'USB0::0x05E6::0x2230::9102515::INSTR', 'ASRL6::INSTR', 'ASRL8::INSTR']

supply = "a"
boxes = "a"
exposition= 0.1


input_states_one = [(1,), (2,), (3,), (4,)]
input_states_two_full = [(1,2), (1,3), (1,4), (2,3), (2,4), (3,4)]
firstNeighbourList = input_states_two_full
inputsFull = input_states_one + input_states_two_full

numParams = len(addresses) * 2
currentParamsTrainable = np.random.rand(numParams) * 64

In [9]:
currentParamsTrainable

array([43.2632115 , 58.95946206, 20.11511193,  2.23455896, 52.22235838,
       20.4915174 , 10.34715041, 40.67117458,  8.59139952,  3.41660253,
       27.58264586, 11.453551  , 36.1902271 , 25.28746835, 52.90407644,
       50.0246164 , 22.84541859, 30.66434466, 14.38172487, 30.91805327])

In [13]:
numParams

20

In [11]:
# lista dei target

targetList = []
targetList.append(np.load("dati_test/conf_03/singles_distribution/b.npz")["distribution"])
targetList.append(np.load("dati_test/conf_03/singles_distribution/c.npz")["distribution"])
targetList.append(np.load("dati_test/conf_03/singles_distribution/d.npz")["distribution"])
targetList.append(np.load("dati_test/conf_03/singles_distribution/e.npz")["distribution"])
targetList.append(np.load("dati_test/conf_03/Couple_coincidences_distributions/bc.npz")["distribution"])
targetList.append(np.load("dati_test/conf_03/Couple_coincidences_distributions/bd.npz")["distribution"])
targetList.append(np.load("dati_test/conf_03/Couple_coincidences_distributions/be.npz")["distribution"])
targetList.append(np.load("dati_test/conf_03/Couple_coincidences_distributions/cd.npz")["distribution"])
targetList.append(np.load("dati_test/conf_03/Couple_coincidences_distributions/ce.npz")["distribution"])
targetList.append(np.load("dati_test/conf_03/Couple_coincidences_distributions/de.npz")["distribution"])

for i in range(len(targetList)):
    targetList[i] = targetList[i].flatten()

targetSingles = np.array(targetList[0:4])
targetDoubles = np.array(targetList[4:])


In [12]:
# Mockup della funzione di misura di Whitlib, usato per testare le funzioni. Non girare quando si usa la libreria Whitelib su device.

def data_collection(inputs: list, Voltages: list, supply, n_supplies: int, boxes, exposition = 0.1, duration = 60, repetitions_singles=1, repetitions_doubles = 2) -> list:
    tempList = []
    for i in range(len(inputs)):
        tempList.append(potatoDict[inputs[i]])
    tempArray = np.array(tempList)
    return tempArray

In [13]:
# Parametri per la funzione mockup. Non girare quando si usa la libreria Whitelib su device.

potatoList = []
potatoList.append(np.load("dati_test/conf_05/singles_distribution/b.npz")["distribution"])
potatoList.append(np.load("dati_test/conf_05/singles_distribution/c.npz")["distribution"])
potatoList.append(np.load("dati_test/conf_05/singles_distribution/d.npz")["distribution"])
potatoList.append(np.load("dati_test/conf_05/singles_distribution/e.npz")["distribution"])
potatoList.append(np.load("dati_test/conf_05/Couple_coincidences_distributions/bc.npz")["distribution"])
potatoList.append(np.load("dati_test/conf_05/Couple_coincidences_distributions/bd.npz")["distribution"])
potatoList.append(np.load("dati_test/conf_05/Couple_coincidences_distributions/be.npz")["distribution"])
potatoList.append(np.load("dati_test/conf_05/Couple_coincidences_distributions/cd.npz")["distribution"])
potatoList.append(np.load("dati_test/conf_05/Couple_coincidences_distributions/ce.npz")["distribution"])
potatoList.append(np.load("dati_test/conf_05/Couple_coincidences_distributions/de.npz")["distribution"])
53
for i in range(len(potatoList)):
    potatoList[i] = potatoList[i].flatten()

keys = [(1,), (2,), (3,), (4,), (1,2), (1,3), (1,4), (2,3), (2,4), (3,4)]

# zip the two lists together to create a list of key-value pairs
key_value_pairs = zip(inputsFull, potatoList)

# convert the list of key-value pairs to a dictionary
potatoDict = dict(key_value_pairs)


In [14]:
# Check parametri iniziali
print(currentParamsTrainable)

[43.2632115  58.95946206 20.11511193  2.23455896 52.22235838 20.4915174
 10.34715041 40.67117458  8.59139952  3.41660253 27.58264586 11.453551
 36.1902271  25.28746835 52.90407644 50.0246164  22.84541859 30.66434466
 14.38172487 30.91805327]


In [15]:
# parametri di training modificabili.

parameterValueMin = 0
parameterValueMax = 64
parameterValueMaxReset = 62
parameterValueMinReset = 2
#shotSize1 = int(1e4)
#shotSize2 = int(1e4)
duration=60
repetitions_singles=1
repetitions_doubles=2
#typeTraining = "proportional"
typeTraining = "absolute"
#typeOrder = "allRandom"
typeOrder = "listRandom"
useTwoPhotons = True
LR = 0.05
#LR = 0.005
#LR = 0.01
LR_check = LR
#LR_move = LR * 5
LR_move = LR
epochsNum = 100
trainingRepetitions = 10
printProgress = "all"                     # "off", "last", "all"
checkPairsNum = 6
avoidBoundary = True
 
supply = "a"
Nsupp = 10
boxes = "a"
exposition= 0.1

### end of tweakable parameters
trainingParams = {"epochsNum" : epochsNum, "LR_check" : LR_check, "LR_move" : LR_move, "useTwoPhotons" : useTwoPhotons, "typeTraining" : typeTraining, "typeOrder" : typeOrder, "printProgress" : printProgress, "checkPairsNum" : checkPairsNum, "firstNeighbourList": firstNeighbourList, "avoidBoundary": avoidBoundary, "supply": supply, "Nsupp": Nsupp, "boxes": boxes, "exposition": exposition, "parameterValueMin": parameterValueMin, "parameterValueMax": parameterValueMax, "parameterValueMinReset": parameterValueMinReset, "parameterValueMaxReset": parameterValueMaxReset}

In [16]:
# Training loop

currentParamsTrainable, lossHistory, bestParams, bestLoss = myTrainingLoopExp(currentParamsTrainable, duration, repetitions_singles, repetitions_doubles, numParams, input_states_one, targetSingles, input_states_two_full, targetDoubles, trainingParams)

Epoch: 0
Current loss is: 1.3564411016059832     Changed param: 2
[5 0 2 4 3 1]
Changing parameter value does not improve loss
Epoch: 1
Current loss is: 1.3564411016059832     Changed param: 0
[3 1 4 0 2 5]
Changing parameter value does not improve loss
Epoch: 2
Current loss is: 1.3564411016059832     Changed param: 7
[1 0 5 2 3 4]
Changing parameter value does not improve loss
Epoch: 3
Current loss is: 1.3564411016059832     Changed param: 5
[4 2 0 3 1 5]
Changing parameter value does not improve loss
Epoch: 4
Current loss is: 1.3564411016059832     Changed param: 19
[3 2 4 0 5 1]
Changing parameter value does not improve loss
Epoch: 5
Current loss is: 1.3564411016059832     Changed param: 4
[0 5 3 1 2 4]
Changing parameter value does not improve loss
Epoch: 6
Current loss is: 1.3564411016059832     Changed param: 14
[4 3 1 2 0 5]
Changing parameter value does not improve loss
Epoch: 7
Current loss is: 1.3564411016059832     Changed param: 11
[2 4 1 5 3 0]
Changing parameter value doe

In [17]:
# print final trained parameters.

print(currentParamsTrainable)

[43.2632115  58.95946206 20.11511193  2.23455896 52.22235838 20.4915174
 10.34715041 40.67117458  8.59139952  3.41660253 27.58264586 11.453551
 36.1902271  25.28746835 52.90407644 50.0246164  22.84541859 30.66434466
 14.38172487 30.91805327]


# Zona test

In [18]:
data_collection(inputs, Voltages, supply, len(addresses), boxes, exposition= 0.1, duration=60, repetitions_singles=1, repetitions_doubles=2)

In [5]:
potatoTemp = np.load("128_auto_calibration/dati_test/Couple_coincidences_distributions/bc.npz")
potato = potatoTemp["distribution"]
potato

array([[   0,  132, 1613, ...,  685,    0,    0],
       [ 132,    0,  432, ...,  392,    0,    0],
       [1613,  432,    0, ..., 1191,    0,    0],
       ...,
       [ 685,  392, 1191, ...,    0,    0,    0],
       [   0,    0,    0, ...,    0,    0,    0],
       [   0,    0,    0, ...,    0,    0,    0]], dtype=int64)

In [6]:
potatoTemp = np.load("128_auto_calibration/matrici/Unitary_mat_01.npz")
potatoMatrix = potatoTemp["U"]
potatoMatrix

array([[8.76579314e-02+0.j        , 6.30672997e-18+0.10299672j,
        8.98995957e-02+0.j        , 1.98685026e-02+0.j        ],
       [6.03864796e-02+0.j        , 6.54446924e-02+0.j        ,
        1.15526753e-01+0.j        , 1.23917989e-01+0.j        ],
       [3.05695134e-02+0.j        , 1.43968593e-01+0.j        ,
        3.45914830e-02+0.j        , 3.47812778e-02+0.j        ],
       [6.78238027e-02+0.j        , 6.93126048e-02+0.j        ,
        1.88943189e-01+0.j        , 3.05194823e-02+0.j        ],
       [1.06669300e-01+0.j        , 8.48091731e-02+0.j        ,
        5.93714407e-02+0.j        , 3.86232129e-02+0.j        ],
       [0.00000000e+00+0.j        , 2.70879428e-02+0.j        ,
        6.84437775e-02+0.j        , 2.14315388e-02+0.j        ],
       [0.00000000e+00+0.j        , 0.00000000e+00+0.j        ,
        0.00000000e+00+0.j        , 0.00000000e+00+0.j        ],
       [0.00000000e+00+0.j        , 0.00000000e+00+0.j        ,
        0.00000000e+00+0.j       

In [7]:
print(np.shape(potato))
print(np.shape(potatoMatrix))

(128, 128)
(128, 4)


In [ ]:
np.shape(np.dot(potatoMatrix, np.transpose(np.conjugate(potatoMatrix))))

In [9]:
(1./128)*np.abs(np.trace(np.dot(potatoMatrix, np.transpose(np.conjugate(potatoMatrix)))))

0.031249999999999997

In [10]:
from scipy.stats import unitary_group
x = unitary_group.rvs(128)


In [11]:
(1./128)*np.abs(np.trace(np.dot(x, np.transpose(np.conjugate(x)))))

1.0

In [12]:
print(potato[60,20])
print(potato[20,60])

2806
2806


In [13]:
print(potato[5,5])

0


In [182]:
len(input_states_one)

4

In [35]:
potato = data_collection(input_states_one, np.sqrt(currentParamsTrainable), supply, Nsupp, boxes, exposition, duration, repetitions_singles, repetitions_doubles)

In [36]:
potato[0]

array([ 2791622,  2834422,  3516378,   429334, 11200284,   612390,
              0,        0,        0,        0,        0,        0,
         773380,  1022589,   166808,  1508211,   410595,  3437176,
         737534,  1022671,  8152880,  3303868,  2818873,   547123,
        3875776,  5833525,   393794, 10053455,  2523080,  4193828,
        2173471,  2274726,  7367548,  4757196,  4004824,  3263883,
        9831439,   557059,  3387000,  1782374,  2403567,  2651357,
        1823102,  3737659,  1454630,  2037804,   820802,   212085,
        4476666,  7167851,  1038132,  4566657,   479369,  1847828,
        4220423,  5374652,  4997814,  2201439,  1063235,  1549915,
        1214921,  7131858,  5027059,  3410856,  4869487,  2290539,
        9259540,  2547235,  5496428,   979428,  1397193,   847250,
        1539274,  2849112,   643519,  1486644,  2384885,  4215197,
        3572521, 12462572,  4548030,  2066391,  2743325,  1161084,
         519874,   712794,  6289700,  4494725,  1184864,  5078

In [43]:
 potato[0]/np.sum(potato[0])

array([0.00804628, 0.00816964, 0.01013524, 0.00123747, 0.03228254,
       0.00176509, 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.00222911, 0.0029474 , 0.00048079,
       0.00434711, 0.00118346, 0.00990696, 0.00212579, 0.00294764,
       0.02349902, 0.00952273, 0.00812483, 0.00157697, 0.01117113,
       0.01681395, 0.00113503, 0.02897704, 0.00727226, 0.01208785,
       0.00626459, 0.00655643, 0.02123546, 0.01371165, 0.01154309,
       0.00940748, 0.02833712, 0.00160561, 0.00976234, 0.00513733,
       0.00692779, 0.007642  , 0.00525472, 0.01077304, 0.00419267,
       0.00587355, 0.00236579, 0.00061129, 0.01290308, 0.02065987,
       0.0029922 , 0.01316246, 0.00138168, 0.00532599, 0.01216451,
       0.01549134, 0.01440518, 0.0063452 , 0.00306456, 0.00446731,
       0.00350176, 0.02055613, 0.01448947, 0.0098311 , 0.0140353 ,
       0.00660201, 0.02668874, 0.00734189, 0.01584233, 0.002823  ,
       0.00402712, 0.00244203, 0.00443664, 0.00821198, 0.00185

In [44]:
np.sum( potato[0]/np.sum(potato[0]))

1.0

In [11]:
potatoArr1 = np.load("matrici/Unitary_mat_01.npz")
potatoArr2 = np.load("matrici/Unitary_mat_02.npz")
potatoUnitary1 = potatoArr1["U"]
potatoUnitary2 = potatoArr2["U"]

In [12]:
np.shape(potatoUnitary1)

(128, 4)

In [13]:
potatoUnitary1

array([[8.76579314e-02+0.j        , 6.30672997e-18+0.10299672j,
        8.98995957e-02+0.j        , 1.98685026e-02+0.j        ],
       [6.03864796e-02+0.j        , 6.54446924e-02+0.j        ,
        1.15526753e-01+0.j        , 1.23917989e-01+0.j        ],
       [3.05695134e-02+0.j        , 1.43968593e-01+0.j        ,
        3.45914830e-02+0.j        , 3.47812778e-02+0.j        ],
       [6.78238027e-02+0.j        , 6.93126048e-02+0.j        ,
        1.88943189e-01+0.j        , 3.05194823e-02+0.j        ],
       [1.06669300e-01+0.j        , 8.48091731e-02+0.j        ,
        5.93714407e-02+0.j        , 3.86232129e-02+0.j        ],
       [0.00000000e+00+0.j        , 2.70879428e-02+0.j        ,
        6.84437775e-02+0.j        , 2.14315388e-02+0.j        ],
       [0.00000000e+00+0.j        , 0.00000000e+00+0.j        ,
        0.00000000e+00+0.j        , 0.00000000e+00+0.j        ],
       [0.00000000e+00+0.j        , 0.00000000e+00+0.j        ,
        0.00000000e+00+0.j       

In [16]:
print((1/2) * np.sum(np.abs(potatoUnitary1 - potatoUnitary2)))

7.189519323446617
